# 06buildaccess: the 15-minute map (Stage 2, part 4 of 4)

3D hexagons: **height** = people living there (as in Stage 1), **colour** = how many of the five everyday
services are within a 15-minute walk. Hover a hexagon for the minutes to each service.

- Input: `hex_access.parquet`, `access_summary.json`, `districts.geojson`, `output/stage1_population.html` (from 02buildmap)
- Output: `output/stage2_access.html`, and for GitHub Pages: `docs/index.html` (this map) and `docs/population.html` (the Stage 1 map, with a link back)

All visual settings are in Section 0, marked `# <-`.

## Section 0: settings

In [36]:
from pathlib import Path

BASE = Path.cwd()
PROCESSED = BASE / "data" / "processed"
OUTPUT = BASE / "output"
DOCS = BASE / "docs"
OUT_HTML = OUTPUT / "stage2_access.html"
STAGE1_HTML = OUTPUT / "stage1_population.html"

# ---- Text ----
PAGE_TITLE = "Klang Valley Reach and Risk: 15-minute walk"                 # <- browser tab
TITLE = "Can you walk to everyday needs?"                                  # <- panel heading
SUBTITLE = ("Colour: how many of five everyday services (clinic, school, park, grocery, rail station) "
            "are within a 15-minute walk along real streets and footpaths. Height: people living there.")   # <-
NOTES = [                                                                   # <- small print, one line each
    "Walking at 4.8 km/h, so 15 minutes = 1.2 km, from the centre of each hexagon (about 0.74 km\u00b2).",
    "Services and footpaths from OpenStreetMap, which misses some places, especially private clinics outside Kuala Lumpur.",
    "Kontur underestimates Putrajaya's population (about 8,600 vs about 110,000 in the 2020 census).",
]
CREDIT = ("Data: \u00a9 OpenStreetMap contributors (ODbL), OSM as of {osm_date}; Kontur Population 2023-11-01 (CC BY 4.0); "
          "Rapid Rail and KTM Komuter stations. Grid: H3. Basemap: \u00a9 CARTO.")   # <-
LINK_TEXT = "See where people live (Stage 1)"                          # <-
BACK_LINK_TEXT = "See the 15-minute walking map (Stage 2)"                # <- added to the Stage 1 page

# ---- Basemap and camera (same as Stage 1) ----
MAP_STYLE = "road"          # <- "dark", "light" or "road"
VIEW_LAT = 3.0              # <-
VIEW_LON = 101.6            # <-
VIEW_ZOOM = 10.0            # <-
VIEW_PITCH = 40             # <-
VIEW_BEARING = -15          # <-

# ---- Hexagon colours: score 0 to 5 ----
SCORE_COLOURS = ["#b2182b", "#e0745a", "#f6c2a6", "#c5dbea", "#6aa6cf", "#2166ac"]   # <- 0 (none) to 5 (all)

# ---- Hexagon shape (same as Stage 1) ----
ELEVATION_SCALE = 0.3       # <- metres of height per person
COVERAGE = 0.88             # <-
OPACITY = 0.90              # <-
DISTRICT_LINE_COLOUR = [60, 60, 60, 140]     # <-

# ---- Title panel (same as Stage 1) ----
PANEL_BG = "rgba(17, 17, 20, 0.82)"          # <-
PANEL_TEXT = "#f2f2f2"                       # <-
PANEL_MUTED = "#a8a8b3"                      # <-
LINK_COLOUR = "#8ec1ff"                      # <-

In [37]:
import json, shutil
import numpy as np
import pandas as pd
import geopandas as gpd
import pydeck as pdk

acc = pd.read_parquet(PROCESSED / "hex_access.parquet")
summary = json.loads((PROCESSED / "access_summary.json").read_text())
districts = gpd.read_file(PROCESSED / "districts.geojson").to_crs(4326)
CATEGORIES = ["healthcare", "school", "park", "grocery", "rail"]
LABELS = {"healthcare": "Clinic or hospital", "school": "School", "park": "Park or playground",
          "grocery": "Grocery", "rail": "Rail station"}
print(len(acc), "hexagons | OSM as of", summary["osm_date"])

4003 hexagons | OSM as of 2026-10-06


## Step 1: colours and tooltip text

In [38]:
def hex_to_rgb(code):
    code = code.lstrip("#")
    return [int(code[i:i + 2], 16) for i in (0, 2, 4)]

def minutes(x):
    if pd.isna(x):
        return "over 75 min"
    return "under 1 min" if x < 1 else f"{x:.0f} min"

alpha = int(OPACITY * 255)
data = acc[acc["population"] > 0].copy()

# VISUAL: colour by score
data["color"] = [hex_to_rgb(SCORE_COLOURS[s]) + [alpha] for s in data["score"]]

# Tooltip text
data["place"] = data["district"].str.replace("W.P. ", "", regex=False)
data["pop_txt"] = data["population"].map("{:,}".format)
data["details"] = data.apply(
    lambda r: "<br/>".join(f"{'&#10003;' if r[c + '_min'] <= summary['threshold_min'] else '&#10005;'} "
                           f"{LABELS[c]}: {minutes(r[c + '_min'])}" for c in CATEGORIES), axis=1)
data = data[["h3", "population", "color", "place", "pop_txt", "score", "details"]]
data.head(3)

,h3,population,color,place,pop_txt,score,details
1,8865054593fffff,1,"[178, 24, 43, 229]",Gombak,1,0,&#10005; Clinic or hospital: 66 min<br/>&#1000...
2,8865054519fffff,1298,"[178, 24, 43, 229]",Gombak,"1,298",0,&#10005; Clinic or hospital: 38 min<br/>&#1000...
4,886505460dfffff,7258,"[106, 166, 207, 229]",Gombak,"7,258",4,&#10003; Clinic or hospital: 5 min<br/>&#10003...


## Step 2: layers, camera, tooltip, save

In [39]:
# VISUAL: 3D hexagons
hex_layer = pdk.Layer("H3HexagonLayer", data, get_hexagon="h3", get_fill_color="color", extruded=True, get_elevation="population", elevation_scale=ELEVATION_SCALE, coverage=COVERAGE, pickable=True, auto_highlight=True)
# VISUAL: district outlines
district_layer = pdk.Layer("GeoJsonLayer", json.loads(districts[["district", "geometry"]].to_json()), stroked=True, filled=False, get_line_color=DISTRICT_LINE_COLOUR, line_width_min_pixels=1)

view = pdk.ViewState(latitude=VIEW_LAT, longitude=VIEW_LON, zoom=VIEW_ZOOM, pitch=VIEW_PITCH, bearing=VIEW_BEARING)

# VISUAL: tooltip
tooltip = {"html": "<b>{place}</b> &middot; {pop_txt} people<br/><b>{score} of 5</b> within 15 minutes' walk<br/>{details}",
           "style": {"backgroundColor": "#1b1b22", "color": "#f2f2f2", "fontSize": "13px",
                     "fontFamily": "Segoe UI, Arial, sans-serif", "padding": "8px 10px", "lineHeight": "1.5"}}

deck = pdk.Deck(layers=[district_layer, hex_layer], initial_view_state=view, map_provider="carto", map_style=MAP_STYLE, tooltip=tooltip)
deck.to_html(str(OUT_HTML), notebook_display=False)
print("Saved", OUT_HTML.name)

Saved stage2_access.html


## Step 3: title panel with legend and key figures

In [40]:
share = summary["share_by_score"]
within = summary["share_within"]

# VISUAL: legend rows, 5 at the top, each with the share of people
legend_rows = "".join(
    f'<div class="row"><span class="sw" style="background:{SCORE_COLOURS[s]}"></span>'
    f'<span class="lab">{s} of 5</span><span class="bar"><span style="width:{share.get(str(s), 0) * 3}px;'
    f'background:{SCORE_COLOURS[s]}"></span></span><span class="pct">{share.get(str(s), 0):.0f}% of people</span></div>'
    for s in range(5, -1, -1))

facts = " &middot; ".join(f"{LABELS[c].split(' ')[0].lower()} {within[c]:.0f}%"
                          for c in sorted(CATEGORIES, key=lambda c: -within[c]))
notes = "".join(f'<div class="small">{t}</div>' for t in NOTES)

def panel_html(link_href, link_text):
    return f'''
<style>
  #kr-panel {{ position:absolute; top:12px; left:12px; z-index:10; max-width:min(400px, calc(100vw - 48px));
    background:{PANEL_BG}; color:{PANEL_TEXT}; padding:14px 16px; border-radius:8px;
    font-family:"Segoe UI", Arial, sans-serif; font-size:13px; line-height:1.45; }}
  #kr-panel h1 {{ font-size:24px; margin:0 0 4px; font-weight:600; }}
  #kr-panel .sub {{ color:{PANEL_MUTED}; margin-bottom:10px; }}
  #kr-panel .row {{ display:flex; align-items:center; gap:8px; margin:3px 0; }}
  #kr-panel .sw {{ width:14px; height:14px; border-radius:3px; flex:none; }}
  #kr-panel .lab {{ width:38px; flex:none; }}
  #kr-panel .bar {{ width:80px; flex:none; }} #kr-panel .bar span {{ display:block; height:8px; border-radius:2px; }}
  #kr-panel .pct {{ color:{PANEL_MUTED}; }}
  #kr-panel .facts {{ margin-top:10px; }}
  #kr-panel .small {{ color:{PANEL_MUTED}; font-size:11px; margin-top:6px; }}
  #kr-panel a {{ color:{LINK_COLOUR}; }}
  @media (max-width: 600px) {{ #kr-panel {{ font-size:12px; padding:10px 12px; }} #kr-panel .small, #kr-panel .sub {{ display:none; }} }}
</style>
<div id="kr-panel">
  <h1>{TITLE}</h1>
  <div class="sub">{SUBTITLE}</div>
  {legend_rows}
  <div class="facts">Within 15 minutes' walk: {facts} of people.</div>
  {notes}
  <div class="small">{CREDIT.format(osm_date=summary["osm_date"])}</div>
  <div class="lnk" style="font-size:12px; margin-top:8px;"><a href="{link_href}">{link_text}</a></div>
</div>
'''

html = OUT_HTML.read_text(encoding="utf-8")
html = html.replace("<title>pydeck</title>", f"<title>{PAGE_TITLE}</title>", 1)
html = html.replace("</body>", panel_html("population.html", LINK_TEXT) + "\n</body>", 1)
OUT_HTML.write_text(html, encoding="utf-8")
print("Saved", OUT_HTML.name, "|", round(OUT_HTML.stat().st_size / 1024), "KB")

Saved stage2_access.html | 1594 KB


## Step 4: pages for GitHub Pages

- `docs/index.html`: this map (the first page visitors see)
- `docs/population.html`: the Stage 1 map, with a link back to this one

In [41]:
DOCS.mkdir(exist_ok=True)
shutil.copyfile(OUT_HTML, DOCS / "index.html")

if STAGE1_HTML.exists():
    s1 = STAGE1_HTML.read_text(encoding="utf-8")
    back = (f'<div class="lnk" style="font-size:12px; margin-top:8px;">'
            f'<a href="index.html" style="color:{LINK_COLOUR}">{BACK_LINK_TEXT}</a></div>')
    s1 = s1.replace("</div>\n</div>", f"</div>\n  {back}\n</div>", 1) if back not in s1 else s1
    (DOCS / "population.html").write_text(s1, encoding="utf-8")
else:
    print("output/stage1_population.html not found: run 02buildmap first")

for f in sorted(DOCS.glob("*.html")):
    print(f.name, round(f.stat().st_size / 1024), "KB")

index.html 1594 KB
population.html 1014 KB


In [42]:
import webbrowser
webbrowser.open((DOCS / "index.html").resolve().as_uri())

True

___
**END**